# ML Drought Forecasting

I am going to be predicting spi6 three months ahead using 12 months of historical climate conditions. While doing this, I am comparing 3 different ML models against a climatological baseline in order to see which is most effective (and also therefore highlighting what type of relationship drought has with the variables) and which is less so. The models being created are as follows:

- Ridge Regression  
- Random Forest 
- Gradient Boosting

No deep learning models have been selected, as there is really not any practical point with a dataset size so small. The reasons for the selected model are below:

**Ridge Regression**    
- Baseline linear model, if it works the variable relationship to drought is simple and linear. 
- Will immediately make it clear if random forest/gradient boosting are going to be better models.  

**Random Forest**   
- Robust and works well with small datasets.    
- Shows non linear relationships between the variables and drought, showing if variables are working together for prediction.   
- Shows feature importance, will highlight which variables and data points are most important in prediction.    

**Gradient Boosting**   
- Will notice more subtle patterns for non linear relationships that random forest cannot see.  
- Learning rate is an easy tool I can tweak to control complexity of prediction.    

There are going to be two baselines for the model to have to beat. The two baselines are climatological and also persistence, if it can beat both it concretely shows that the model is good enough to predict future drought conditions. These baselines are:

**Climatological Baseline**
- Question: "I am predicting the historical average every time".    
- Test: Can the model beat this prediction knowing nothing about the conditions during the prediction date. 

**Persistence Baseline**
- Question: "I am predicting that SPI6 in 3 months will be identical to what it is today".     
- Test: Can the model beat the assumption that absolutely nothing is going to change.   

The final part of the model creation and forecasting is temporal split. In order to avoid the model ever seeing the data it is being tested on during training, I am going to split it in two. To maximise the amount of training data available to the model, **1980-2019** will all be training data, and then 2020-2024 will be used to test the model and to run predictions upon.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

df = pd.read_csv("../data/processed/mendoza_basin_with_indices.csv", parse_dates=["time"], index_col="time")

print(f"Loaded {len(df)} months")
print(f"NaN counts:\n{df.isnull().sum()}")

# Feature Config
target = "spi6"
horizon = 3
lookback = 12
features = [
    "precip_mm",
    "temp_c",
    "pev_mm",
    "runoff_mm",
    "soil_moisture_0_7cm",
    "spi3",
    "spi6",
]

Loaded 540 months
NaN counts:
precip_mm               0
temp_c                  0
pev_mm                  0
runoff_mm               0
soil_moisture_0_7cm     0
spi3                    2
spi6                    5
spi12                  11
spei3                   2
spei6                   5
dtype: int64


## Feature Engineering

I transform the dataset into a supervised learning problem. For every prediction, I use 12 months of lookback across 6 variables as features, and also cyclical month encoding to capture seasonality.

In [2]:
df_clean = df.dropna()
print(f"Months available: {len(df_clean)}")

def create_supervised_dataset(data, target_col, features_cols, horizon, lookback):
    x_rows = []
    y_rows = []
    dates = []
    col_names = []

    for i in range(lookback, len(data) - horizon):
        window = data.iloc[i - lookback:i][features_cols].values
        features = window.flatten()
        current_month = data.index[i].month
        month_sin = np.sin(2 * np.pi * current_month / 12)
        month_cos = np.cos(2 * np.pi * current_month / 12)
        features = np.append(features, [month_sin, month_cos])
        target = data.iloc[i + horizon][target_col]

        x_rows.append(features)
        y_rows.append(target)
        dates.append(data.index[i + horizon])
    
    for lag in range(lookback, 0, -1):
        for ftre in features_cols:
            col_names.append(f"{ftre}_lag{lag}")
    col_names.extend(["month_sin", "month_cos"])
    
    x = pd.DataFrame(x_rows, columns=col_names, index=dates)
    y = pd.Series(y_rows, index=dates, name=target_col)

    return x, y

x, y = create_supervised_dataset(df_clean, target, features, horizon, lookback)

print("Supervised dataser:")
print(f"    Samples:  {len(x)}")
print(f"    Features: {x.shape[1]}")
print(f"    Feature names head: {list(x.columns[:5])}")
print(f"    Feature names tail: {list(x.columns[-5:])}")
print(f"    Target range: {y.min():.2f} to {y.max():.2f}")


Months available: 529
Supervised dataser:
    Samples:  514
    Features: 74
    Feature names head: ['precip_mm_lag12', 'temp_c_lag12', 'pev_mm_lag12', 'runoff_mm_lag12', 'soil_moisture_0_7cm_lag12']
    Feature names tail: ['runoff_mm_lag1', 'soil_moisture_0_7cm_lag1', 'spi3_lag1', 'month_sin', 'month_cos']
    Target range: -2.04 to 2.70


## Training and Testing Split

In order to keep good temporal space, the training and testing data are complete separate so that the model never sees the future data. The training data is going to be **1980-2019**, and the testing data is going to be **2020-2024**.

In [16]:
test_years = 5
cutoff = x.index.max() - pd.DateOffset(years=test_years)

x_train = x[x.index <= cutoff]
y_train = y[y.index <= cutoff]
x_test = x[x.index > cutoff]
y_test = y[y.index > cutoff]

scaler = StandardScaler()
x_trained_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)

## Model Training and Evaluation

Comparing three models against a climatological baseline and a persistence baseline. The skill score measures improvement over this baseline, a positive value means the model adds value beyond simple historical averages.

In [17]:
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {
        "Model": model_name,
        "MAE": round(mae, 3),
        "RMSE": round(rmse, 3),
        "R²": round(r2, 3),
    }

# Climatological Baseline
clim_pred = np.full(len(y_test), y_train.mean())
clim_metrics = evaluate_model(y_test, clim_pred, "Climatology Baseline")
clim_rmse = clim_metrics["RMSE"]

# Persistence Baseline
persistence_pred = []
for target_date in y_test.index:
    forecast_date = target_date - pd.DateOffset(months=horizon)

    if forecast_date in df_clean:
        persistence_pred.append(df_clean.loc[forecast_date, target])
    else:
        closest_idx = df_clean.index.get_indexer([forecast_date], method="nearest")[0]
        persistence_pred.append(df_clean.iloc[closest_idx][target])

persistence_pred = np.array(persistence_pred)
persist_metrics = evaluate_model(y_test, persistence_pred, "Persistence Baseline")
persist_rmse = persist_metrics["RMSE"]

print(f"Climatology baseline RMSE: {clim_rmse}")
print(f"Persistence baseline RMSE: {persist_rmse}")

models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=5,
        random_state=40,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        max_depth=10,
        learning_rate=0.05,
        min_samples_leaf=5,
        random_state=40,
    ),
}

all_metrics = []
predictions = {}

for name, pred, metrics in [("Climatology Baseline", clim_pred, clim_metrics), ("Persistence Baseline", persistence_pred, persist_metrics)]:
    metrics["Skill vs Clim"] = round(1 - (metrics["RMSE"] / clim_rmse), 3)
    metrics["Skill vs Persist"] = round(1 - (metrics["RMSE"] / persist_rmse), 3)
    all_metrics.append(metrics)
    predictions[name] = pred

for name, model in models.items():
    print(f"\nTraining the model {name}")

    if "Ridge" in name:
        model.fit(x_trained_scaled, y_train)
        y_pred = model.predict(x_test_scaled)
    else:
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)
    
    metrics = evaluate_model(y_test, y_pred, name)
    metrics["Skill vs Clim"] = round(1 - (metrics["RMSE"] / clim_rmse), 3)
    metrics["Skill vs Persist"] = round(1 - (metrics["RMSE"] / persist_rmse), 3)
    all_metrics.append(metrics)
    predictions[name] = y_pred

    print(f"  R²: {metrics['R²']}, Skill vs Climatological: {metrics['Skill vs Clim']}, Skill vs Persistence: {metrics['Skill vs Persist']}")

results_df = pd.DataFrame(all_metrics)
print("\n                                Model Comparison")
print(results_df.to_string(index=False))

Climatology baseline RMSE: 1.264
Persistence baseline RMSE: 0.859

Training the model Ridge Regression
  R²: -1.763, Skill vs Climatological: 0.008, Skill vs Persistence: -0.46

Training the model Random Forest
  R²: -0.263, Skill vs Climatological: 0.329, Skill vs Persistence: 0.013

Training the model Gradient Boosting
  R²: -0.249, Skill vs Climatological: 0.333, Skill vs Persistence: 0.019

                                Model Comparison
               Model   MAE  RMSE     R²  Skill vs Clim  Skill vs Persist
Climatology Baseline 1.122 1.264 -1.807          0.000            -0.471
Persistence Baseline 0.681 0.859 -0.298          0.320             0.000
    Ridge Regression 1.093 1.254 -1.763          0.008            -0.460
       Random Forest 0.720 0.848 -0.263          0.329             0.013
   Gradient Boosting 0.730 0.843 -0.249          0.333             0.019
